In [1]:
import pandas as pd
import numpy as np
import re
import string
import difflib

Load the raw data

In [2]:
# Load the data
df = pd.read_csv("../data/raw/indigenous-business/bcindigenousbusinesslistings.csv")

Inspecting the data

In [3]:
# Inspect the data
print(df.info())
print(df.head())
print(f"Initial number of rows: {len(df)}") 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Business Name        1259 non-null   object 
 1   Description          1135 non-null   object 
 2   Web Site             699 non-null    object 
 3   City                 1258 non-null   object 
 4   Latitude             1258 non-null   float64
 5   Longitude            1258 non-null   float64
 6   Keywords             1257 non-null   object 
 7   Region               1259 non-null   object 
 8   Type                 1123 non-null   object 
 9   Industry Sector      1222 non-null   object 
 10  Year Formed          648 non-null    float64
 11  Number of Employees  572 non-null    object 
dtypes: float64(3), object(9)
memory usage: 118.2+ KB
None
                                       Business Name  \
0                                Ellipsis Energy Inc   
1  Indigenous Communit

Column Name Standardization

In [4]:
#  Clean column names (convert to lowercase and replace spaces with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

Remove Unnecessary Columns

In [5]:
# Remove unnecessary columns
columns_to_drop = ['description', 'web_site', 'keywords']
df = df.drop(columns=columns_to_drop, errors='ignore')

Removal of Duplicates

In [6]:
# Remove duplicate rows
df = df.drop_duplicates()

In [7]:
# check no of rows after removing duplicates
print(f"No of rows after removing duplicates: {len(df)}") 

No of rows after removing duplicates: 1259


Critical Data Validation

In [8]:
# Remove rows missing critical information
#business_name is a mandatory field here
if 'business_name' in df.columns:
    df = df[df['business_name'].notna() & (df['business_name'] != '')]

In [9]:
# check no of rows after removing rows missing critical information
print(f"No of rows: {len(df)}") 

No of rows: 1259


Ensure Year is an integer

In [10]:
# Ensure year_formed is a nullable integer
df['year_formed'] = pd.to_numeric(df['year_formed'], errors='coerce').astype('Int64')

Cleanup industry_sector

In [11]:
# custom function to standardize industry_sector data
def clean_industry_sector(sector):
    if pd.isna(sector):
        return np.nan
    
    # Convert to string
    sector = str(sector).strip()
    
    # Handle cases starting with colon
    if sector.startswith(':'):
        sector = sector[1:].strip()
    
    # Remove ALL number patterns including:
    # "23 - ", "44-45 - ", "1.5 - ", "54 – " (with en dash)
    sector = re.sub(r'^[\d\.]+\s*[-–—]?\s*[\d\.]*\s*[-–—]\s*', '', sector).strip()
    
    # Return np.nan if empty, otherwise capitalize first letter
    return np.nan if not sector else sector[0].upper() + sector[1:]


print("test cleaning:")
test_case = ":54 – Professional, scientific and technical services"
print(f"'{test_case}' → '{clean_industry_sector(test_case)}'")


df['industry_sector'] = df['industry_sector'].apply(clean_industry_sector)

test cleaning:
':54 – Professional, scientific and technical services' → 'Professional, scientific and technical services'


Data Formatting

In [12]:
# Trim whitespace in string fields
text_cols = ['business_name', 'city', 'industry_sector','region','type']
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

Clean the Region Categories (Combine "Vancouver Island And Coast" and "Vancouver Island / Coast" to "Vancouver Island / Coast")

In [13]:
# check current region column categories
df['region'].value_counts()

region
Lower Mainland / Southwest    343
Vancouver Island / Coast      278
Thompson / Okanagan           193
North Coast                   169
Northeast                      86
Nechako                        76
Cariboo                        58
Kootenay                       41
Vancouver Island and Coast     15
Name: count, dtype: int64

In [14]:
# replace 'Vancouver Island and Coast' with 'Vancouver Island / Coast'
df.loc[df['region'] == 'Vancouver Island and Coast', 'region'] = 'Vancouver Island / Coast'

In [15]:
# check region column categories after value replacement
df['region'].value_counts()

region
Lower Mainland / Southwest    343
Vancouver Island / Coast      293
Thompson / Okanagan           193
North Coast                   169
Northeast                      86
Nechako                        76
Cariboo                        58
Kootenay                       41
Name: count, dtype: int64

Clean City Names

In [16]:
# Normalize city names:
# Remove '.', ',', and 'BC' (Ex. 'Fort St. John, B.C.' to 'Fort St John')
# Apply capwords to non-nan values (Ex. "HUDSON'S HOPE" to "Hudon's Hope")
# Convert spaces longer than one space to one space (Ex. 'Fort  St John' to 'Fort St John')
# Remove any leading or trailing spaces (Ex. ' Chilliwack ' to 'Chilliwack')

df['city'] = df['city'].str.replace('.', '')
df['city'] = df['city'].str.replace(',', '')
df['city'] = df['city'].str.replace('BC', '')
df['city'] = df['city'].apply(
    lambda x: string.capwords(x) if isinstance(x, str) else x
)
df["city"] = df["city"].apply(
    lambda x: re.sub(r"\s+", " ", x) if isinstance(x, str) else x
)
df['city'] = df['city'].str.strip()

In [17]:
# Identify groups of city names that are similar to each other to identify spelling errors
# (note: I could not find a good list of BC municipalities, and some 'cities' in 
# this data may not qualify as a municipality)

cities_list = sorted(list(df['city'][df['city'].notna()].unique())) # alphabetical list of non-nan cities

matches = [] # accumulator for groups of city names that are similar to each other

for city in cities_list:
    # identify similar city names for the current city
    match = difflib.get_close_matches(
        city,
        cities_list,
        cutoff=0.75 # threshold of similarity
    )
    
    # alphabetize the group
    match = sorted(match)

    # add the group to the list (matches) if the group is larger than one
    # and isn't already in the list
    if len(match) > 1 and match not in matches:
        matches.append(match)

matches

[['100 Mile House', '150 Mile House'],
 ['Abbotsfford', 'Abbotsford'],
 ['Brurns Lake', 'Burns Lake'],
 ['Cambell River', 'Campbell River'],
 ['Christina Lake', 'Nitinat Lake'],
 ['Coquitlam', 'Cquitlam', 'Port Coquitlam'],
 ['Cowichan', 'Cowichan Bay'],
 ['Coquitlam', 'Cquitlam'],
 ['Dawson Creek', 'Dog Creek'],
 ['Dease Lake', 'Fraser Lake'],
 ['Fort St John', 'Ft St John'],
 ['Hazelton', 'New Hazelton', 'Old Hazelton'],
 ['Lilloet', 'Lillooet', 'Lilooet'],
 ['Lytton', 'Lyyton'],
 ['Masset', 'Massett', 'Old Masset'],
 ['Masset', 'Massett'],
 ['Merrit', 'Merritt'],
 ['North Vancouver', 'Vancouver', 'West Vancouver'],
 ['Masset', 'Old Masset'],
 ['Penticon', 'Penticton'],
 ['Coquitlam', 'Port Coquitlam'],
 ['Port Edward', 'Port Hardy'],
 ['Shalalth', 'Shalath'],
 ['Telegraph Cove', 'Telegraph Creek'],
 ['Ucluelet', 'Uculet'],
 ['West Bank', 'Westbank'],
 ['Wonowon', 'Wonowoon']]

In [18]:
# Manually replace spelling errors with the correct spellling

# 'Abbotsfford' → 'Abbotsford'
df.loc[df['city'] == 'Abbotsfford', 'city'] = 'Abbotsford'

# 'Brurns Lake' → 'Burns Lake'
df.loc[df['city'] == 'Brurns Lake', 'city'] = 'Burns Lake'

# 'Cambell River' → 'Campbell River'
df.loc[df['city'] == 'Cambell River', 'city'] = 'Campbell River'

# 'Cquitlam' → 'Coquitlam'
df.loc[df['city'] == 'Cquitlam', 'city'] = 'Coquitlam'

# 'Ft St John' → 'Fort St John'
df.loc[df['city'] == 'Ft St John', 'city'] = 'Fort St John'

# 'Lilloet' and 'Lilooet' → 'Lillooet'
df.loc[df['city'] == 'Lilloet', 'city'] = 'Lillooet'
df.loc[df['city'] == 'Lilooet', 'city'] = 'Lillooet'

# 'Lyyton' → 'Lytton'
df.loc[df['city'] == 'Lyyton', 'city'] = 'Lytton'

# 'Massett' → 'Masset'
df.loc[df['city'] == 'Massett', 'city'] = 'Masset'

# 'Merrit' → 'Merritt'
df.loc[df['city'] == 'Merrit', 'city'] = 'Merritt'

# 'Penticon' → 'Penticton'
df.loc[df['city'] == 'Penticon', 'city'] = 'Penticton'

# 'Shalath' → 'Shalalth'
df.loc[df['city'] == 'Shalath', 'city'] = 'Shalalth'

# 'Ucluelet' and 'Uculet' → 'Ucluelet'
df.loc[df['city'] == 'Ucluelet', 'city'] = 'Ucluelet'
df.loc[df['city'] == 'Uculet', 'city'] = 'Ucluelet'

# 'West Bank' → 'Westbank'
df.loc[df['city'] == 'West Bank', 'city'] = 'Westbank'

# 'Wonowoon' → 'Wonowon'
df.loc[df['city'] == 'West Bank', 'city'] = 'Westbank'

Clean Ownership Types

In [19]:
# check current non-nan ownership types
sorted(list(df['type'][df['type'].notna()].unique()))

[': Community Owned Company',
 'Community Owned',
 'Community Owned Company',
 'Development Corporation',
 'Joint Venture',
 'Partnership',
 'Partnershp',
 'Private Company']

In [20]:
# ': Community Owned Company' and 'Community Owned' → 'Community Owned Company'
df.loc[df['type'] == ': Community Owned Company', 'type'] = 'Community Owned Company'
df.loc[df['type'] == 'Community Owned', 'type'] = 'Community Owned Company'

# 'Partnershp' → 'Partnership'
df.loc[df['type'] == 'Partnershp', 'type'] = 'Partnership'

In [21]:
# check non-nan ownership types after replacement
sorted(list(df['type'][df['type'].notna()].unique()))

['Community Owned Company',
 'Development Corporation',
 'Joint Venture',
 'Partnership',
 'Private Company']

Clean Number of Employees

In [22]:
# check current non-nan values
sorted(list(df['number_of_employees'][df['number_of_employees'].notna()].unique()))

[' ',
 '1 to 4',
 '1 to 4 ',
 '10 to 19',
 '100 to 199',
 '100 to 199 ',
 '20 to 49',
 '200 to 499',
 '5 to 9',
 '50 to 99',
 '500 plus',
 '55 to 99']

In [23]:
# convert ' ' to nan
df['number_of_employees'] = df['number_of_employees'].replace(' ', np.nan)

# strip leading and trailing spaces
df['number_of_employees'] = df['number_of_employees'].str.strip()

# '55 to 99' → '50 to 99'
df.loc[df['number_of_employees'] == '55 to 99', 'number_of_employees'] = '50 to 99'

In [24]:
# check non-nan values after replacement
sorted(list(df['number_of_employees'][df['number_of_employees'].notna()].unique()))

['1 to 4',
 '10 to 19',
 '100 to 199',
 '20 to 49',
 '200 to 499',
 '5 to 9',
 '50 to 99',
 '500 plus']

Save the Cleaned data

In [25]:
# Save cleaned data
df.to_csv("../data/clean/bcindigenousbiz.csv", index=False)

Validation of Cleaned Data

In [26]:
# Validate cleaned data
clean_data= pd.read_csv("../data/clean/bcindigenousbiz.csv")
print(f"Final cleaned dataset rows: {len(clean_data)}")  # Final row count
clean_data.head()

Final cleaned dataset rows: 1259


,business_name,city,latitude,longitude,region,type,industry_sector,year_formed,number_of_employees
0,Ellipsis Energy Inc,Moberly Lake,55.819370,-121.834602,Northeast,Private Company,"Mining, quarrying, and oil and gas extraction",2012.0,5 to 9
1,Indigenous Community Development & Prosperity ...,Enderby,50.551498,-119.133546,Thompson / Okanagan,Private Company,Other services (except public administration),2020.0,1 to 4
2,Formline Construction Ltd.,Burnaby,49.266050,123.005840,Lower Mainland / Southwest,Private Company,Construction,2021.0,1 to 4
3,Quilakwa Investments Ltd.,Enderby,50.537507,-119.141955,Thompson / Okanagan,Community Owned Company,Accommodation and food services,1984.0,20 to 49
4,Quilakwa Esso,Enderby,50.537507,-119.141955,Thompson / Okanagan,Community Owned Company,Retail trade,1984.0,10 to 19
